# Python System Environment — The Raise of Python

One shared Python, two users, three projects.

## Meet Alice and Bob

> **Alice:** Chocolate, ticket **IT-2048**, and an Ubuntu workspace request for IrisLab.
>
> **Bob:** Fine. One shared system Python. It has always been enough.

Alice and Bob work together at the same company. One of the company's business domains is starting work on *IrisLab*, and Alice is a software engineer on its team. IrisLab is a large project spanning many tools and areas of the Python ecosystem. At its core, IrisLab provides a machine-learning model that detects eye color from images. Around that core, the project includes a REST API, an end-user CLI, and security middleware. Bob is the grumpy, old-fashioned IT administrator. Run every cell in order inside the disposable Ubuntu 24.04 [Dev Container](./README.md), starting with a fresh container. It supplies Bob and Alice accounts, Python 3.12, and separate Jupyter tooling; package downloads need network access.

## Create the Three Projects

A dependency is a package an application needs; `==` requires an exact version. Each project has only `requirements.txt` (its dependencies) and `app.py` (its version check).

| Project directory | Owner | Dependencies |
|---|---|---|
| `projects/legacy_app/` | Bob | `requests==2.25.1` |
| `projects/fastapi_app_1/` | Alice | `requests==2.32.3`, `fastapi==0.68.1` |
| `projects/fastapi_app_2/` | Alice | `requests==2.32.3`, `fastapi==0.111.1` |

### Write Requirements

`!bash` runs Linux commands from the notebook. `mkdir -p` creates directories; `printf` writes lines into files with `>`; a trailing `\` continues the command.

In [ ]:
!bash -c 'mkdir -p projects/legacy_app projects/fastapi_app_1 projects/fastapi_app_2'
!bash -c 'printf "%s\n" "requests==2.25.1" > projects/legacy_app/requirements.txt'
!bash -c 'printf "%s\n" "requests==2.32.3" "fastapi==0.68.1" > projects/fastapi_app_1/requirements.txt'
!bash -c 'printf "%s\n" "requests==2.32.3" "fastapi==0.111.1" > projects/fastapi_app_2/requirements.txt'


### Write Minimal Apps

Each app imports its packages and prints `MATCH` or `CONFLICT` against its required versions. Conflicts are printed rather than raised as errors so the lab can continue.

In [ ]:
!bash -c 'printf "%s\n" \
  "import requests" \
  "" \
  "print(\"Legacy App\")" \
  "print(\"requests:\", requests.__version__, \"MATCH\" if requests.__version__ == \"2.25.1\" else \"CONFLICT: expected 2.25.1\")" \
  > projects/legacy_app/app.py'


In [ ]:
!bash -c 'printf "%s\n" \
  "import requests" \
  "import fastapi" \
  "" \
  "print(\"FastAPI App 1\")" \
  "print(\"requests:\", requests.__version__, \"MATCH\" if requests.__version__ == \"2.32.3\" else \"CONFLICT: expected 2.32.3\")" \
  "print(\"fastapi:\", fastapi.__version__, \"MATCH\" if fastapi.__version__ == \"0.68.1\" else \"CONFLICT: expected 0.68.1\")" \
  > projects/fastapi_app_1/app.py'


In [ ]:
!bash -c 'printf "%s\n" \
  "import requests" \
  "import fastapi" \
  "" \
  "print(\"FastAPI App 2\")" \
  "print(\"requests:\", requests.__version__, \"MATCH\" if requests.__version__ == \"2.32.3\" else \"CONFLICT: expected 2.32.3\")" \
  "print(\"fastapi:\", fastapi.__version__, \"MATCH\" if fastapi.__version__ == \"0.111.1\" else \"CONFLICT: expected 0.111.1\")" \
  > projects/fastapi_app_2/app.py'


## Stage 1 — Shared System Packages

> **Bob:** Install everything once. Simple.
>
> **Alice:** Let's try your Legacy App first.

System-wide packages are available to both users. Ubuntu's PEP 668 protection marks Python as externally managed by the OS; `--break-system-packages` overrides that protection for this experiment, only inside this disposable container.

### Inspect the Shared Python

`PATH` determines which command the shell runs; `sys.path` determines where Python searches for imports. Find the shell's `python3` command:

In [ ]:
!bash -c 'command -v python3'


Check the system interpreter's Python version:

In [ ]:
!bash -c '/usr/bin/python3 --version'


Show Bob's interpreter with `sys.executable`. `sudo -H -u` selects a user and their home directory; `-s` excludes user packages for this system-only stage.

In [ ]:
!bash -c 'sudo -H -u bob /usr/bin/python3 -s -c "import sys; print(\"Bob:\", sys.executable)"'


Check that Alice uses the same interpreter:

In [ ]:
!bash -c 'sudo -H -u alice /usr/bin/python3 -s -c "import sys; print(\"Alice:\", sys.executable)"'


List package locations and the import search path with `python -m site`:

In [ ]:
!bash -c '/usr/bin/python3 -s -m site'


Both users run `/usr/bin/python3`. The output includes two shared package locations:

- `/usr/lib/python3/dist-packages`: Ubuntu/APT packages (managed by Ubuntu's package manager).
- `/usr/local/lib/python3.12/dist-packages`: administrator-installed packages used in this experiment.

### Install and Run Bob’s App

`python -m pip` runs that interpreter's package installer; `-r` reads a requirements file. Here, `sudo` installs into the shared directory.

In [ ]:
!bash -c 'sudo /usr/bin/python3 -m pip install --quiet --break-system-packages -r projects/legacy_app/requirements.txt'


In [ ]:
!bash -c 'sudo -H -u bob /usr/bin/python3 -s projects/legacy_app/app.py'
!bash -c '/usr/bin/python3 -s -c "import requests; print(requests.__file__)"'


Legacy App reports `requests: 2.25.1 MATCH`. Its package comes from the shared directory.

### Show the Shared System Conflict

Bob installs FastAPI App 1's requirements into the same location, then both users run their apps.

In [ ]:
!bash -c 'sudo /usr/bin/python3 -m pip install --quiet --break-system-packages -r projects/fastapi_app_1/requirements.txt'


In [ ]:
!bash -c 'sudo -H -u bob /usr/bin/python3 -s projects/legacy_app/app.py'
!bash -c 'sudo -H -u alice /usr/bin/python3 -s projects/fastapi_app_1/app.py'


Alice's app matches, but Bob now sees `requests: 2.32.3 CONFLICT: expected 2.25.1`. One shared installation cannot provide both required versions; reinstalling Bob's version would instead conflict with Alice's requirement.

> **Bob:** You upgraded my package!
>
> **Alice:** Our package. That's the problem.

## Stage 2 — User-Scoped Packages

> **Bob:** Then keep your packages in your own home directory.
>
> **Alice:** Good for users. Let's test projects too.

`pip install --user` installs under the selected user's home directory. Ubuntu 24.04 requires the PEP 668 override here too.

### Inspect User Package Directories

Look at `USER_SITE`, each user's package directory. It may say “doesn't exist” until the first user installation.

In [ ]:
!bash -c 'sudo -H -u bob /usr/bin/python3 -m site'
!bash -c 'sudo -H -u alice /usr/bin/python3 -m site'


Bob's target starts with `/home/bob/.local/`; Alice's starts with `/home/alice/.local/`. They still use the same system interpreter.

### Install for Bob and Alice

For these first user installs, `--ignore-installed` ensures each user gets their own copies, even when a requirement is already satisfied globally. User packages take priority over shared packages when these apps run.

In [ ]:
!bash -c 'sudo -H -u bob /usr/bin/python3 -m pip install --quiet --user --ignore-installed --break-system-packages -r projects/legacy_app/requirements.txt'


In [ ]:
!bash -c 'sudo -H -u alice /usr/bin/python3 -m pip install --quiet --user --ignore-installed --break-system-packages -r projects/fastapi_app_1/requirements.txt'


In [ ]:
!bash -c 'sudo -H -u bob /usr/bin/python3 projects/legacy_app/app.py'
!bash -c 'sudo -H -u alice /usr/bin/python3 projects/fastapi_app_1/app.py'
!bash -c 'sudo -H -u bob /usr/bin/python3 -c "import requests; print(\"Bob:\", requests.__file__)"'
!bash -c 'sudo -H -u alice /usr/bin/python3 -c "import requests, fastapi; print(\"Alice:\", requests.__file__, fastapi.__file__, sep=\"\\n\")"'


Both apps now match. The printed paths confirm their packages live in separate home directories.

### Show Alice’s User-Package Conflict

FastAPI App 2 needs a newer FastAPI. Install it as Alice, then run all three projects.

In [ ]:
!bash -c 'sudo -H -u alice /usr/bin/python3 -m pip install --quiet --user --break-system-packages -r projects/fastapi_app_2/requirements.txt'


In [ ]:
!bash -c 'sudo -H -u alice /usr/bin/python3 projects/fastapi_app_1/app.py'
!bash -c 'sudo -H -u alice /usr/bin/python3 projects/fastapi_app_2/app.py'
!bash -c 'sudo -H -u bob /usr/bin/python3 projects/legacy_app/app.py'


FastAPI App 1 sees `0.111.1` instead of `0.68.1`; App 2 matches. Bob still has `requests==2.25.1`: **`pip install --user` separates users, not projects.**

> **Bob:** At least mine still works. Your two apps can argue.
>
> **Alice:** Or each can have its own environment.

## Stage 3 — Virtual Environments

> **Bob:** Three more Python installations? Really?
>
> **Alice:** Three package directories, using the same base Python.

A virtual environment is a project-specific Python environment with its own packages. By default, it excludes system-wide and user-installed packages.

### Create One Environment per Project

`venv` is included with Python; `.venv` is the directory we choose for each environment.

In [ ]:
!bash -c '/usr/bin/python3 -m venv projects/legacy_app/.venv'
!bash -c '/usr/bin/python3 -m venv projects/fastapi_app_1/.venv'
!bash -c '/usr/bin/python3 -m venv projects/fastapi_app_2/.venv'


### Install Dependencies Separately

Use each environment's interpreter to install into that environment. No PEP 668 override is needed.

In [ ]:
!bash -c 'projects/legacy_app/.venv/bin/python -m pip install --quiet -r projects/legacy_app/requirements.txt'
!bash -c 'projects/legacy_app/.venv/bin/python projects/legacy_app/app.py'


In [ ]:
!bash -c 'projects/fastapi_app_1/.venv/bin/python -m pip install --quiet -r projects/fastapi_app_1/requirements.txt'
!bash -c 'projects/fastapi_app_1/.venv/bin/python projects/fastapi_app_1/app.py'


In [ ]:
!bash -c 'projects/fastapi_app_2/.venv/bin/python -m pip install --quiet -r projects/fastapi_app_2/requirements.txt'


### Inspect Environments and Run All Three Apps

`sys.prefix` is the current environment's directory; `sys.base_prefix` identifies the base Python installation. `readlink -f` follows each interpreter's link to its underlying executable.

In [ ]:
!bash -c 'readlink -f /usr/bin/python3'
!bash -c 'for project in legacy_app fastapi_app_1 fastapi_app_2; do \
  readlink -f projects/$project/.venv/bin/python; \
  projects/$project/.venv/bin/python -c "import sys, requests; print(\"interpreter:\", sys.executable); print(\"prefix:\", sys.prefix); print(\"base:\", sys.base_prefix); print(\"package:\", requests.__file__)"; \
done'


All three interpreter links lead to the same system Python, with base prefix `/usr`. Each environment has a different prefix and its own `.venv/lib/python3.12/site-packages` directory.

In [ ]:
!bash -c 'projects/legacy_app/.venv/bin/python projects/legacy_app/app.py'
!bash -c 'projects/fastapi_app_1/.venv/bin/python projects/fastapi_app_1/app.py'
!bash -c 'projects/fastapi_app_2/.venv/bin/python projects/fastapi_app_2/app.py'


All five version checks report `MATCH`. Installing App 2's newer FastAPI left App 1 and Legacy App unchanged: all three projects coexist without dependency conflicts.

> **Bob:** Fine. Keep the environments. And the ticket.
>
> **Alice:** You can keep the chocolate.

## Final Comparison

| Installation method | Isolates users? | Isolates projects? |
|---|---|---|
| System-wide installation | No | No |
| `pip install --user` | Yes | No |
| Virtual environment | Yes, when used separately | Yes |

**System Python is shared. User installations separate users. Virtual environments separate projects.**